# From Draft to Living Cell

_Investigation `draft-to-living-cell` — coder reproduction notebook._

**Question.** How far can a cell be built compositionally — assembled from typed interfaces,
each specified as a *draft* before any mechanism is chosen, then compiled into
something that actually runs — and does that composition hold together into a
whole cell that lives and dies?

This investigation turns *A meta-modeler's guide to the cellular interface* into an executable test of compositional modeling. Each figure of the paper is written twice. First as a draft: a typed, unit-bearing biological interface and a behavioral contract, with no committed mechanism — a promise about what a process couples to and does, not how. Then it is compiled, which installs one or more conforming mechanisms — mechanistic, rule-based, data-driven, or otherwise — behind the identical ports, so the running simulation preserves the declared interface exactly.

The point is not to assemble a single monolithic cell model. It is to test whether a biological model can remain open-ended as its mechanisms change, its assumptions fail, and its level of description shifts. Interfaces provide the places where models can be connected, isolated, replaced, or expanded; graph rewrites allow the composition itself to change through events such as growth, division, and disintegration.

The investigation therefore follows the paper's central claim: composition is not the final architecture of a model, but an ongoing way of building one. Across the studies below, we ask whether the same biological specification can support alternative mechanisms, whether those mechanisms can be composed without losing their interface semantics, and whether the resulting system can reorganize as the biology demands. The endpoint is a composed whole cell that grows, divides, loses viability, and changes its own level of description — demonstrating the full arc from interface specification to living, changing simulation.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/meta-modelers-guide/meta-modelers-guide').is_dir():
    REPO = Path('/home/runner/work/meta-modelers-guide/meta-modelers-guide')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from meta_modelers_guide.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: The Typed Interface (`typed-interface`)

**Question.** Can the cellular boundary be specified as nothing but a set of typed, unit-bearing exchange ports (chemical, mechanical, electrical, thermal, plus growth rate, shape, objective, viability) with no committed mechanism, and then compiled — by installing one conforming handler — into a running, bounded, goal-directed cell whose interface is exactly the one declared?

**Claim.** A cell can be specified at one level by its interface — typed exchange variables such as chemical flux, force, current, heat, growth, shape, objective, and viability — without committing to the mechanism that realizes those behaviors.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `interaction-modalities` | `meta_modelers_guide.composites.fig04a-interaction-modalities` | 0 | — |
| `cellular-interface` | `meta_modelers_guide.composites.fig04b-cellular-interface` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig04a-interaction-modalities`** — `spec_meta_modelers_guide_composites_fig04a_interaction_modalities` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig04a_interaction_modalities = load_spec(REPO / 'meta_modelers_guide/composites/fig04a-interaction-modalities.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig04a_interaction_modalities)

In [ ]:
# === Edit parameters for composite 'Interaction Modalities' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'nutrient_exchange'  (local:NutrientExchange)
spec_meta_modelers_guide_composites_fig04a_interaction_modalities['state']['nutrient_exchange']['config']['interval'] = 1.0

# process 'motile_force'  (local:MotileForce)
spec_meta_modelers_guide_composites_fig04a_interaction_modalities['state']['motile_force']['config']['interval'] = 1.0

# process 'growth'  (local:Growth)
spec_meta_modelers_guide_composites_fig04a_interaction_modalities['state']['growth']['config']['interval'] = 1.0

# process 'electrical_signaling'  (local:ElectricalSignaling)
spec_meta_modelers_guide_composites_fig04a_interaction_modalities['state']['electrical_signaling']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig04b-cellular-interface`** — `spec_meta_modelers_guide_composites_fig04b_cellular_interface` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig04b_cellular_interface = load_spec(REPO / 'meta_modelers_guide/composites/fig04b-cellular-interface.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig04b_cellular_interface)

In [ ]:
# === Edit parameters for composite 'The Cellular Interface' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'cell'  (local:CellularInterface)
spec_meta_modelers_guide_composites_fig04b_cellular_interface['state']['cell']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: typed-interface ===
STUDY = 'typed-interface'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**typed-interface-dynamics**


In [ ]:
# typed-interface-dynamics
show_viz(_render_one('image:visualizations/typed-interface-dynamics.svg', {'chart': 'image', 'caption': 'Dynamics of the compiled Fig 4 executable, run to completion.'}, RUNS_DB, STUDY_YAML))

**fig04a-interaction-modalities**


In [ ]:
# fig04a-interaction-modalities
show_viz(_render_one('image:visualizations/fig04a-interaction-modalities.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig04b-cellular-interface**


In [ ]:
# fig04b-cellular-interface
show_viz(_render_one('image:visualizations/fig04b-cellular-interface.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig04-illustration**


In [ ]:
# fig04-illustration
show_viz(_render_one('image:visualizations/fig04-illustration.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| shape-growth | kind=observable path=shape expr=max(shape) | op >= value 2.0 provenance executable run: shape 1.0→4.2 (chemical uptake drives growth); exec last≈4.2 |
| chemical-uptake | kind=observable path=chemical expr=min(chemical) | op <= value -0.5 provenance chemical flux 0.0→-0.8 (net uptake across the interface); exec last≈-0.8 |
| objective-climbs | kind=observable path=objective expr=last(objective) | op >= value 1.0 provenance objective 0.0→1.6 as growth proceeds; exec last≈1.6 |
| draft-is-inert | kind=observable path=shape expr=max(shape) | op >= value 2.0 provenance inert-draft run: shape stays 1.0 (no mechanism → no growth); the control the correct model is contrasted against |


## Study: Closing the Loop (`closing-the-loop`)

**Question.** Does the cell–environment coupling of Fig 5 close into a genuine sense/act loop when the environment is a real spatial field — i.e. does the cell read a diffusing chemical field, act back on it through an uptake flux, and grow from what it takes up, all over one shared field store?

**Claim.** Sensing and acting are one coupling, not two: a cell and its environment can be specified as a shared interface over which the cell reads a field and acts back on it.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `cell-environment` | `meta_modelers_guide.composites.fig05-cell-environment` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig05-cell-environment`** — `spec_meta_modelers_guide_composites_fig05_cell_environment` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig05_cell_environment = load_spec(REPO / 'meta_modelers_guide/composites/fig05-cell-environment.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig05_cell_environment)

In [ ]:
# === Edit parameters for composite 'Cell–Environment Coupling' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'reaction_diffusion'  (local:ReactionDiffusion)
spec_meta_modelers_guide_composites_fig05_cell_environment['state']['reaction_diffusion']['config']['interval'] = 1.0

# process 'production_degradation'  (local:ProductionDegradation)
spec_meta_modelers_guide_composites_fig05_cell_environment['state']['production_degradation']['config']['interval'] = 1.0

# process 'mechanical_stress'  (local:MechanicalStress)
spec_meta_modelers_guide_composites_fig05_cell_environment['state']['mechanical_stress']['config']['interval'] = 1.0

# process 'single_cell_processes'  (local:SingleCellProcesses)
spec_meta_modelers_guide_composites_fig05_cell_environment['state']['single_cell_processes']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: closing-the-loop ===
STUDY = 'closing-the-loop'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**closing-the-loop-dynamics**


In [ ]:
# closing-the-loop-dynamics
show_viz(_render_one('image:visualizations/closing-the-loop-dynamics.svg', {'chart': 'image', 'caption': 'Dynamics of the compiled Fig 5 executable, run to completion.'}, RUNS_DB, STUDY_YAML))

**fig05-cell-environment**


In [ ]:
# fig05-cell-environment
show_viz(_render_one('image:visualizations/fig05-cell-environment.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig05-illustration**


In [ ]:
# fig05-illustration
show_viz(_render_one('image:visualizations/fig05-illustration.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cell-draws-well | kind=observable path=uptake_flux expr=last(uptake_flux) | op >= value 0.2 provenance cell takes up from the field — uptake flux 0.0→0.34, bolus cell drawn down 1.0→0.2 (exec last≈0.34) |
| cell-acts-back | kind=observable path=traction expr=last(traction) | op >= value 0.3 provenance cell acts back mechanically — traction 0.0→0.41 (exec last≈0.41) |
| mechanical-response | kind=observable path=mechanical_field expr=last(mechanical_field) | op >= value 0.5 provenance mechanical field driven by the cell 0.0→0.94 (exec last≈0.94) |
| draft-is-inert | kind=observable path=traction expr=last(traction) | op >= value 0.3 provenance inert-draft run stays at seed — traction 0 (no mechanism → no dynamics) |


## Study: One Interface, Three Mechanisms (`one-interface-three-mechanisms`)

**Question.** Can a single metabolism interface — the ports nutrients ⇒ {biomass, energy, entropy, secretions} of Fig 6 — be realized by three genuinely different mechanisms (a lumped-yield process, a saturating-kinetic process, and a real flux-balance optimization via COBRApy) while every other part of the composite, and the interface itself, stays byte-for-byte the same?

**Claim.** One metabolic interface can be realized by many mechanisms — a lumped yield, saturating kinetics, or a real flux-balance solver — each producing its own distinct dynamics, without changing what the interface exposes.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `disintegration` | `meta_modelers_guide.composites.fig06-disintegration` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig06-disintegration`** — `spec_meta_modelers_guide_composites_fig06_disintegration` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig06_disintegration = load_spec(REPO / 'meta_modelers_guide/composites/fig06-disintegration.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig06_disintegration)

In [ ]:
# === Edit parameters for composite 'Cell Disintegration' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'coarse_grained_metabolism'  (local:CoarseGrainedMetabolism)
spec_meta_modelers_guide_composites_fig06_disintegration['state']['coarse_grained_metabolism']['config']['interval'] = 1.0

# process 'catalyzed_reaction_network'  (local:CatalyzedReactionNetwork)
spec_meta_modelers_guide_composites_fig06_disintegration['state']['catalyzed_reaction_network']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: one-interface-three-mechanisms ===
STUDY = 'one-interface-three-mechanisms'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**one-interface-three-mechanisms-dynamics**


In [ ]:
# one-interface-three-mechanisms-dynamics
show_viz(_render_one('image:visualizations/one-interface-three-mechanisms-dynamics.svg', {'chart': 'image', 'caption': 'One interface, three handlers — three distinct trajectories (coarse 4.0 / kinetic 2.667 / FBA 6.29, acetate overflow 30.4), run to completion.'}, RUNS_DB, STUDY_YAML))

**fig06-disintegration**


In [ ]:
# fig06-disintegration
show_viz(_render_one('image:visualizations/fig06-disintegration.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig06-illustration**


In [ ]:
# fig06-illustration
show_viz(_render_one('image:visualizations/fig06-illustration.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| coarse-linear-growth | kind=observable path=biomass expr=last(biomass) | op >= value 3.5 provenance coarse (lumped linear yield) → biomass 4.0 (exec fig06-executable-coarse) |
| kinetic-saturating-growth | kind=observable path=biomass expr=last(biomass) | op >= value 2.0 provenance kinetic (saturating Michaelis–Menten uptake) → biomass 2.667 (exec fig06-executable-kinetic), distinct from the coarse 4.0 |
| fba-overflow-secretion | kind=observable path=secretions expr=last(secretions) | op >= value 10.0 provenance FBA overflow metabolism: acetate on the secretions port 0→30.4 (exec fig06-executable-fba); biomass reaches 6.29 |
| impostor-rejected | kind=observable path=biomass expr=last(biomass) | op >= value 1.0 provenance REJECTION control: NonConformingMetabolism is rejected by the compiler (CompileError names missing biomass/energy/entropy/secretions); it never runs, so no biomass observable exists — MUST fail |


## Study: From Molecules to the Nested Cell (`the-nested-cell`)

**Question.** How do molecules compose into a cell? At the deepest grain a single molecular machine — F1Fo ATP synthase (Fig 7) — is compiled from its draft and run as a PMF-driven rotary catalyst that synthesizes ATP on its chemical output port. That mechanism sits at the bottom of the Fig 8 nested cell — a deeply nested place graph (membrane, cytoplasm, nucleus, chromosome, chromatin, nucleosome) with a gene-expression cascade wired to its deepest leaves. Does compilation preserve every interface, from the lone molecular mechanism up through six levels of nesting?

**Claim.** A molecular mechanism (F1Fo ATP synthase) sits at the deepest grain and feeds a gene-expression cascade nested across six place-graph levels; each interface is preserved, from the lone molecular machine up to the deepest leaf of the nesting.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `molecular-mechanism` | `meta_modelers_guide.composites.fig07-molecular-mechanism` | 0 | — |
| `nested-hierarchy` | `meta_modelers_guide.composites.fig08-nested-hierarchy` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig07-molecular-mechanism`** — `spec_meta_modelers_guide_composites_fig07_molecular_mechanism` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig07_molecular_mechanism = load_spec(REPO / 'meta_modelers_guide/composites/fig07-molecular-mechanism.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig07_molecular_mechanism)

In [ ]:
# === Edit parameters for composite 'A Molecular Mechanism' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'molecular_mechanism'  (local:MolecularMechanism)
spec_meta_modelers_guide_composites_fig07_molecular_mechanism['state']['molecular_mechanism']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig08-nested-hierarchy`** — `spec_meta_modelers_guide_composites_fig08_nested_hierarchy` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig08_nested_hierarchy = load_spec(REPO / 'meta_modelers_guide/composites/fig08-nested-hierarchy.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig08_nested_hierarchy)

In [ ]:
# === Edit parameters for composite 'Nested Molecular Hierarchy' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'transmembrane_transport'  (local:TransmembraneTransport)
spec_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['transmembrane_transport']['config']['interval'] = 1.0

# process 'replication_and_repair'  (local:ReplicationAndRepair)
spec_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['replication_and_repair']['config']['interval'] = 1.0

# process 'cell_metabolism'  (local:CellMetabolism)
spec_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['cell_metabolism']['config']['interval'] = 1.0

# process 'transcription'  (local:Transcription)
spec_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['transcription']['config']['interval'] = 1.0

# process 'translation'  (local:Translation)
spec_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['translation']['config']['interval'] = 1.0

# process 'subunit_assembly'  (local:SubunitAssembly)
spec_meta_modelers_guide_composites_fig08_nested_hierarchy['state']['subunit_assembly']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: the-nested-cell ===
STUDY = 'the-nested-cell'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**the-nested-cell-dynamics**


In [ ]:
# the-nested-cell-dynamics
show_viz(_render_one('image:visualizations/the-nested-cell-dynamics.svg', {'chart': 'image', 'caption': 'Dynamics of the compiled executables, run to completion — the F1Fo ATP synthase molecular mechanism (chemical_out 0→100) at the deepest grain alongside the Fig 8 nested-cell cascade (metabolites, mRNA, energy rising from zero).'}, RUNS_DB, STUDY_YAML))

**fig08-nested-hierarchy**


In [ ]:
# fig08-nested-hierarchy
show_viz(_render_one('image:visualizations/fig08-nested-hierarchy.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig08-illustration**


In [ ]:
# fig08-illustration
show_viz(_render_one('image:visualizations/fig08-illustration.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| atp-synthesized | kind=observable path=chemical_out expr=last(chemical_out) | op >= value 50.0 provenance exec last≈100 — F1Fo ATP synthase (ATP synthesized on chemical_out 0→100, exec fig07-executable) |
| metabolites-produced | kind=observable path=metabolites expr=last(metabolites) | op >= value 1.0 provenance metabolism feeds the cascade, producing metabolites (exec last≈1.5) |
| mrna-transcribed | kind=observable path=rna expr=last(rna) | op >= value 0.05 provenance gene transcribed to mRNA (exec last≈0.18) |
| draft-is-inert | kind=observable path=metabolites expr=last(metabolites) | op >= value 1.0 provenance inert-draft run stays at seed (no mechanism → no dynamics) |


## Study: Self-Made (`self-made`)

**Question.** How does a cell hold itself together? Does the Fig 9 composition express autopoiesis — metabolism, containment, and replication mutually producing one another — and does that same closure appear when each function is realized at a coarse, a self-organized, or a molecular grain?

**Claim.** A cell holds itself together through closure: metabolism, containment, and replication each produce what the others need, so the organization sustains itself.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `coarse-graining` | `meta_modelers_guide.composites.fig09a-coarse-graining` | 0 | — |
| `minimal-cell` | `meta_modelers_guide.composites.fig09b-minimal-cell` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig09a-coarse-graining`** — `spec_meta_modelers_guide_composites_fig09a_coarse_graining` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig09a_coarse_graining = load_spec(REPO / 'meta_modelers_guide/composites/fig09a-coarse-graining.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig09a_coarse_graining)

In [ ]:
# === Edit parameters for composite 'Self-Organization & Coarse-Graining' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'metabolism_closure'  (local:MetabolismClosure)
spec_meta_modelers_guide_composites_fig09a_coarse_graining['state']['metabolism_closure']['config']['interval'] = 1.0

# process 'autocatalysis'  (local:Autocatalysis)
spec_meta_modelers_guide_composites_fig09a_coarse_graining['state']['autocatalysis']['config']['interval'] = 1.0

# process 'containment_closure'  (local:ContainmentClosure)
spec_meta_modelers_guide_composites_fig09a_coarse_graining['state']['containment_closure']['config']['interval'] = 1.0

# process 'membrane_self_assembly'  (local:MembraneSelfAssembly)
spec_meta_modelers_guide_composites_fig09a_coarse_graining['state']['membrane_self_assembly']['config']['interval'] = 1.0

# process 'lipid_aggregation'  (local:LipidAggregation)
spec_meta_modelers_guide_composites_fig09a_coarse_graining['state']['lipid_aggregation']['config']['interval'] = 1.0

# process 'replication_closure'  (local:ReplicationClosure)
spec_meta_modelers_guide_composites_fig09a_coarse_graining['state']['replication_closure']['config']['interval'] = 1.0

# process 'template_replication'  (local:TemplateReplication)
spec_meta_modelers_guide_composites_fig09a_coarse_graining['state']['template_replication']['config']['interval'] = 1.0

# process 'template_directed_synthesis'  (local:TemplateDirectedSynthesis)
spec_meta_modelers_guide_composites_fig09a_coarse_graining['state']['template_directed_synthesis']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig09b-minimal-cell`** — `spec_meta_modelers_guide_composites_fig09b_minimal_cell` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig09b_minimal_cell = load_spec(REPO / 'meta_modelers_guide/composites/fig09b-minimal-cell.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig09b_minimal_cell)

In [ ]:
# === Edit parameters for composite 'The Minimal Cell' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'minimal_cell_containment'  (local:MinimalCellContainment)
spec_meta_modelers_guide_composites_fig09b_minimal_cell['state']['minimal_cell_containment']['config']['interval'] = 1.0

# process 'minimal_cell_metabolism'  (local:MinimalCellMetabolism)
spec_meta_modelers_guide_composites_fig09b_minimal_cell['state']['minimal_cell_metabolism']['config']['interval'] = 1.0

# process 'gene_expression'  (local:GeneExpression)
spec_meta_modelers_guide_composites_fig09b_minimal_cell['state']['gene_expression']['config']['interval'] = 1.0

# process 'minimal_cell_replication'  (local:MinimalCellReplication)
spec_meta_modelers_guide_composites_fig09b_minimal_cell['state']['minimal_cell_replication']['config']['interval'] = 1.0

# process 'diffusion'  (local:Diffusion)
spec_meta_modelers_guide_composites_fig09b_minimal_cell['state']['diffusion']['config']['interval'] = 1.0

# process 'reactions'  (local:Reactions)
spec_meta_modelers_guide_composites_fig09b_minimal_cell['state']['reactions']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: self-made ===
STUDY = 'self-made'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**self-made-dynamics**


In [ ]:
# self-made-dynamics
show_viz(_render_one('image:visualizations/self-made-dynamics.svg', {'chart': 'image', 'caption': 'Dynamics of the compiled Fig 9 executables (fig09a + fig09b), run to completion.'}, RUNS_DB, STUDY_YAML))

**fig09a-coarse-graining**


In [ ]:
# fig09a-coarse-graining
show_viz(_render_one('image:visualizations/fig09a-coarse-graining.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig09b-minimal-cell**


In [ ]:
# fig09b-minimal-cell
show_viz(_render_one('image:visualizations/fig09b-minimal-cell.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig09-illustration**


In [ ]:
# fig09-illustration
show_viz(_render_one('image:visualizations/fig09-illustration.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig09-illustration-2**


In [ ]:
# fig09-illustration-2
show_viz(_render_one('image:visualizations/fig09-illustration-2.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| membrane-sustains | kind=observable path=membrane expr=last(membrane) | op >= value 1.0 provenance intact closure self-sustains the membrane (exec last≈1.6) |
| enzyme-maintained | kind=observable path=enzymes expr=last(enzymes) | op >= value 0.5 provenance the enzyme pool is held up by the closure (exec last≈0.72) |
| draft-is-inert | kind=observable path=membrane expr=last(membrane) | op >= value 1.0 provenance inert draft stays at seed (no mechanism / no closure) |


## Study: Divide (`divide`)

**Question.** Is cell division in Fig 10 a genuine structural rewrite of the place graph — a single cell node actually becoming two daughter nodes at runtime — rather than a pre-declared post-structure that is merely animated?

**Claim.** Division is a change to the composition itself: one cell becomes two through a graph rewrite that creates new nodes at runtime, not a pre-drawn pair.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `division-rewrite` | `meta_modelers_guide.composites.fig10-1-rewrite` | 0 | — |
| `division` | `meta_modelers_guide.composites.fig10-1-division` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig10-1-rewrite`** — `spec_meta_modelers_guide_composites_fig10_1_rewrite` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig10_1_rewrite = load_spec(REPO / 'meta_modelers_guide/composites/fig10-1-rewrite.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig10_1_rewrite)

In [ ]:
# === Edit parameters for composite 'Cell Division — Live Topology' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_meta_modelers_guide_composites_fig10_1_rewrite['parameters']['cycle']['default'] = 3.0

# process 'cell_cycle'  (local:CellCycleDivision)
spec_meta_modelers_guide_composites_fig10_1_rewrite['state']['cell_cycle']['interval'] = 1.0
spec_meta_modelers_guide_composites_fig10_1_rewrite['state']['cell_cycle']['config']['cycle'] = '${cycle}'

**Composite `meta_modelers_guide.composites.fig10-1-division`** — `spec_meta_modelers_guide_composites_fig10_1_division` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig10_1_division = load_spec(REPO / 'meta_modelers_guide/composites/fig10-1-division.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig10_1_division)

In [ ]:
# === Edit parameters for composite 'Cell Division — Draft Interface' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'dna_replication'  (local:DNAReplication)
spec_meta_modelers_guide_composites_fig10_1_division['state']['dna_replication']['config']['interval'] = 1.0

# process 'segregate_chromosome'  (local:SegregateChromosome)
spec_meta_modelers_guide_composites_fig10_1_division['state']['segregate_chromosome']['config']['interval'] = 1.0

# process 'divide'  (local:Divide)
spec_meta_modelers_guide_composites_fig10_1_division['state']['divide']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: divide ===
STUDY = 'divide'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**divide-dynamics**


In [ ]:
# divide-dynamics
show_viz(_render_one('image:visualizations/divide-dynamics.svg', {'chart': 'image', 'caption': 'Division dynamics, run to completion through the engine.'}, RUNS_DB, STUDY_YAML))

**fig10-1-division**


In [ ]:
# fig10-1-division
show_viz(_render_one('image:visualizations/fig10-1-division.svg', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig10-illustration**


In [ ]:
# fig10-illustration
show_viz(_render_one('image:visualizations/fig10-illustration.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| division-occurs | kind=observable path=cell_count expr=last(cell_count) | op >= value 1.5 provenance lineage divides once (exec cell_count 1→2) |
| daughters-spawn | kind=observable path=dna expr=last(dna) | op >= value 1.0 provenance division spawns two new daughters (exec daughter dna 0→2.75) |
| draft-is-inert | kind=observable path=cell_count expr=last(cell_count) | op >= value 1.5 provenance inert-draft run stays at seed cell_count 1 (no mechanism → no dynamics) |


## Study: Multicellular Rewrites (`multicellular`)

**Question.** Once a single cell divides (Fig 10-1), can the emergence of multicellularity be expressed as compositional topology rewrites — DEVELOPMENT, where cells reorganize into a colony that is itself a higher-level composite with aggregate observables (Fig 10-2), and EVOLUTION, where variation and selection extend the interface alphabet itself by adding a new port to a lineage (Fig 10-3)?

**Claim.** Multicellularity is two compositional topology rewrites at a level above the single cell: DEVELOPMENT reorganizes cells into a colony that is itself a composite with its own aggregate behavior, and EVOLUTION rewrites a population under selection so a lineage gains an entirely new interface port.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `development-rewrite` | `meta_modelers_guide.composites.fig10-2-rewrite` | 0 | — |
| `development` | `meta_modelers_guide.composites.fig10-2-development` | 0 | — |
| `evolution-rewrite` | `meta_modelers_guide.composites.fig10-3-rewrite` | 0 | — |
| `evolution` | `meta_modelers_guide.composites.fig10-3-evolution` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.fig10-2-rewrite`** — `spec_meta_modelers_guide_composites_fig10_2_rewrite` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig10_2_rewrite = load_spec(REPO / 'meta_modelers_guide/composites/fig10-2-rewrite.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig10_2_rewrite)

In [ ]:
# === Edit parameters for composite 'Biofilm Development — Live Topology' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_meta_modelers_guide_composites_fig10_2_rewrite['parameters']['capacity']['default'] = 5
spec_meta_modelers_guide_composites_fig10_2_rewrite['parameters']['grow_every']['default'] = 2.0

# process 'development'  (local:BiofilmDevelopment)
spec_meta_modelers_guide_composites_fig10_2_rewrite['state']['development']['interval'] = 1.0
spec_meta_modelers_guide_composites_fig10_2_rewrite['state']['development']['config']['grow_every'] = '${grow_every}'
spec_meta_modelers_guide_composites_fig10_2_rewrite['state']['development']['config']['capacity'] = '${capacity}'

**Composite `meta_modelers_guide.composites.fig10-2-development`** — `spec_meta_modelers_guide_composites_fig10_2_development` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig10_2_development = load_spec(REPO / 'meta_modelers_guide/composites/fig10-2-development.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig10_2_development)

In [ ]:
# === Edit parameters for composite 'Biofilm Development — Draft Interface' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'surface_attachment'  (local:SurfaceAttachment)
spec_meta_modelers_guide_composites_fig10_2_development['state']['surface_attachment']['config']['interval'] = 1.0

# process 'ecm_secretion'  (local:ECMSecretion)
spec_meta_modelers_guide_composites_fig10_2_development['state']['ecm_secretion']['config']['interval'] = 1.0

# process 'biofilm_growth'  (local:BiofilmGrowth)
spec_meta_modelers_guide_composites_fig10_2_development['state']['biofilm_growth']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig10-3-rewrite`** — `spec_meta_modelers_guide_composites_fig10_3_rewrite` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig10_3_rewrite = load_spec(REPO / 'meta_modelers_guide/composites/fig10-3-rewrite.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig10_3_rewrite)

In [ ]:
# === Edit parameters for composite 'Evolution — Live Topology' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# tunable parameters (filled into ${name} placeholders):
spec_meta_modelers_guide_composites_fig10_3_rewrite['parameters']['founders']['default'] = 3
spec_meta_modelers_guide_composites_fig10_3_rewrite['parameters']['capacity']['default'] = 6
spec_meta_modelers_guide_composites_fig10_3_rewrite['parameters']['generation']['default'] = 2.0
spec_meta_modelers_guide_composites_fig10_3_rewrite['parameters']['mutate_at']['default'] = 4.0

# process 'evolution'  (local:LineageEvolution)
spec_meta_modelers_guide_composites_fig10_3_rewrite['state']['evolution']['interval'] = 1.0
spec_meta_modelers_guide_composites_fig10_3_rewrite['state']['evolution']['config']['generation'] = '${generation}'
spec_meta_modelers_guide_composites_fig10_3_rewrite['state']['evolution']['config']['mutate_at'] = '${mutate_at}'
spec_meta_modelers_guide_composites_fig10_3_rewrite['state']['evolution']['config']['founders'] = '${founders}'
spec_meta_modelers_guide_composites_fig10_3_rewrite['state']['evolution']['config']['capacity'] = '${capacity}'

**Composite `meta_modelers_guide.composites.fig10-3-evolution`** — `spec_meta_modelers_guide_composites_fig10_3_evolution` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig10_3_evolution = load_spec(REPO / 'meta_modelers_guide/composites/fig10-3-evolution.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig10_3_evolution)

In [ ]:
# === Edit parameters for composite 'Evolution — Draft Interface' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'variation'  (local:Variation)
spec_meta_modelers_guide_composites_fig10_3_evolution['state']['variation']['config']['interval'] = 1.0

# process 'selection'  (local:Selection)
spec_meta_modelers_guide_composites_fig10_3_evolution['state']['selection']['config']['interval'] = 1.0

# process 'port_addition'  (local:PortAddition)
spec_meta_modelers_guide_composites_fig10_3_evolution['state']['port_addition']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: multicellular ===
STUDY = 'multicellular'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**multicellular-dynamics**


In [ ]:
# multicellular-dynamics
show_viz(_render_one('image:visualizations/multicellular-dynamics.svg', {'chart': 'image', 'caption': 'Multicellular rewrites, run to completion through the engine. Left panel — DEVELOPMENT (Fig 10-2) — cells attach and grow (cells 1→1.57, attached 0→1.35), secrete ECM (0→1.8), and an aggregate biofilm_mass emerges (0→2.25). Right panel — EVOLUTION (Fig 10-3) — the population grows under selection (cell_count 1→3.4) and a variant lineage acquires a new interface port (new_port 0→0.57).'}, RUNS_DB, STUDY_YAML))

**fig10-illustration-2**


In [ ]:
# fig10-illustration-2
show_viz(_render_one('image:visualizations/fig10-illustration-2.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

**fig10-illustration-3**


In [ ]:
# fig10-illustration-3
show_viz(_render_one('image:visualizations/fig10-illustration-3.png', {'chart': 'image'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| colony-develops | kind=observable path=biofilm_mass expr=last(biofilm_mass) | op >= value 1.5 provenance development rewrite — an aggregate colony observable emerges (exec biofilm_mass last≈2.25) |
| ecm-accumulates | kind=observable path=ecm expr=last(ecm) | op >= value 1.0 provenance the colony secretes a shared extracellular matrix (exec ecm last≈1.8) |
| population-grows | kind=observable path=cell_count expr=last(cell_count) | op >= value 2.0 provenance evolution rewrite — the population grows under selection (exec cell_ecoli.cell_count last≈3.4) |
| capability-emerges | kind=observable path=new_port expr=last(new_port) | op >= value 0.3 provenance a variant lineage acquires a new interface capability (exec new_port last≈0.57) |
| draft-is-inert | kind=observable path=biofilm_mass expr=last(biofilm_mass) | op >= value 1.5 provenance inert-draft run stays at seed (no mechanism → no dynamics, biofilm_mass stays 0) |


## Study: The Living Atlas (`the-living-atlas`)

**Question.** The paper's semantic figures each compile to an executable that runs — but the real test of composition is whether their independently-authored mechanisms share enough interface to COMPOSE. This capstone wires the recurring figure modules (thermal interface, cell↔environment uptake, viability-gated metabolism, the viability monitor, and the division/disintegration rewrites) through shared cell and environment stores and runs them as one whole cell. Do the figures compose into a single cell that grows, divides, and dies?

**Claim.** The paper's figure mechanisms are modular: wired through shared cell and environment stores they COMPOSE into one whole cell that grows (biomass peak ≈5.1), divides once (cell_count 1 → 2), and dies (viability 1.0 → 0.018 after a thermal shock 37 → 50 °C converts biomass to debris 0 → 4.87). Backing it, all twelve of the paper's semantic figures compile to executables that run to completion (12/12) — the gallery whose modules this cell composes.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `whole-cell` | `meta_modelers_guide.composites.whole-cell` | 0 | — |
| `fig06-exec-coarse` | `meta_modelers_guide.composites.fig06-executable-coarse` | 0 | — |
| `fig06-exec-kinetic` | `meta_modelers_guide.composites.fig06-executable-kinetic` | 0 | — |
| `fig04b-exec` | `meta_modelers_guide.composites.fig04b-executable` | 0 | — |
| `fig05-exec-spatial` | `meta_modelers_guide.composites.fig05-executable` | 0 | — |
| `fig06-exec-fba` | `meta_modelers_guide.composites.fig06-executable-fba` | 0 | — |
| `fig07-exec-molecular` | `meta_modelers_guide.composites.fig07-executable` | 0 | — |
| `fig08-exec-hierarchy` | `meta_modelers_guide.composites.fig08-executable` | 0 | — |
| `fig09a-exec-coarse-graining` | `meta_modelers_guide.composites.fig09a-executable` | 0 | — |
| `fig09b-exec-minimal-cell` | `meta_modelers_guide.composites.fig09b-executable` | 0 | — |
| `fig10-1-exec-division` | `meta_modelers_guide.composites.fig10-1-executable` | 0 | — |
| `fig10-2-exec-development` | `meta_modelers_guide.composites.fig10-2-executable` | 0 | — |
| `fig10-3-exec-evolution` | `meta_modelers_guide.composites.fig10-3-executable` | 0 | — |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `meta_modelers_guide.composites.whole-cell`** — `spec_meta_modelers_guide_composites_whole_cell` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_whole_cell = load_spec(REPO / 'meta_modelers_guide/composites/whole-cell.composite.json')
describe_spec(spec_meta_modelers_guide_composites_whole_cell)

In [ ]:
# === Edit parameters for composite 'whole-cell' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'thermal'  (local:ThermalEnvironment)
spec_meta_modelers_guide_composites_whole_cell['state']['thermal']['interval'] = 0.1
spec_meta_modelers_guide_composites_whole_cell['state']['thermal']['config']['temp_normal'] = 37.0
spec_meta_modelers_guide_composites_whole_cell['state']['thermal']['config']['temp_shock'] = 50.0
spec_meta_modelers_guide_composites_whole_cell['state']['thermal']['config']['shock_time'] = 12.0

# process 'uptake'  (local:Uptake)
spec_meta_modelers_guide_composites_whole_cell['state']['uptake']['interval'] = 0.1
spec_meta_modelers_guide_composites_whole_cell['state']['uptake']['config']['uptake_rate'] = 0.5

# process 'metabolism'  (local:ViabilityGatedMetabolism)
spec_meta_modelers_guide_composites_whole_cell['state']['metabolism']['interval'] = 0.1
spec_meta_modelers_guide_composites_whole_cell['state']['metabolism']['config']['k'] = 0.6
spec_meta_modelers_guide_composites_whole_cell['state']['metabolism']['config']['biomass_yield'] = 0.8
spec_meta_modelers_guide_composites_whole_cell['state']['metabolism']['config']['energy_yield'] = 0.4

# process 'monitor'  (local:ViabilityMonitor)
spec_meta_modelers_guide_composites_whole_cell['state']['monitor']['interval'] = 0.1
spec_meta_modelers_guide_composites_whole_cell['state']['monitor']['config']['temp_opt'] = 37.0
spec_meta_modelers_guide_composites_whole_cell['state']['monitor']['config']['temp_tol'] = 5.0
spec_meta_modelers_guide_composites_whole_cell['state']['monitor']['config']['relax'] = 0.5
spec_meta_modelers_guide_composites_whole_cell['state']['monitor']['config']['division_threshold'] = 1.0
spec_meta_modelers_guide_composites_whole_cell['state']['monitor']['config']['viability_floor'] = 0.3

# process 'division'  (local:DivisionEvent)
spec_meta_modelers_guide_composites_whole_cell['state']['division']['interval'] = 0.1

# process 'disintegration'  (local:DisintegrationEvent)
spec_meta_modelers_guide_composites_whole_cell['state']['disintegration']['interval'] = 0.1
spec_meta_modelers_guide_composites_whole_cell['state']['disintegration']['config']['decay_rate'] = 0.4

**Composite `meta_modelers_guide.composites.fig06-executable-coarse`** — `spec_meta_modelers_guide_composites_fig06_executable_coarse` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig06_executable_coarse = load_spec(REPO / 'meta_modelers_guide/composites/fig06-executable-coarse.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig06_executable_coarse)

In [ ]:
# === Edit parameters for composite 'fig06-executable-coarse' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'coarse_grained_metabolism'  (local:CoarseMetabolism)
spec_meta_modelers_guide_composites_fig06_executable_coarse['state']['coarse_grained_metabolism']['config']['biomass_yield'] = 0.5
spec_meta_modelers_guide_composites_fig06_executable_coarse['state']['coarse_grained_metabolism']['config']['energy_yield'] = 0.3
spec_meta_modelers_guide_composites_fig06_executable_coarse['state']['coarse_grained_metabolism']['config']['entropy_rate'] = 0.1
spec_meta_modelers_guide_composites_fig06_executable_coarse['state']['coarse_grained_metabolism']['config']['secretion_frac'] = 0.2
spec_meta_modelers_guide_composites_fig06_executable_coarse['state']['coarse_grained_metabolism']['config']['interval'] = 1.0

# process 'catalyzed_reaction_network'  (local:KineticReactionNetwork)
spec_meta_modelers_guide_composites_fig06_executable_coarse['state']['catalyzed_reaction_network']['config']['k'] = 0.2
spec_meta_modelers_guide_composites_fig06_executable_coarse['state']['catalyzed_reaction_network']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig06-executable-kinetic`** — `spec_meta_modelers_guide_composites_fig06_executable_kinetic` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig06_executable_kinetic = load_spec(REPO / 'meta_modelers_guide/composites/fig06-executable-kinetic.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig06_executable_kinetic)

In [ ]:
# === Edit parameters for composite 'fig06-executable-kinetic' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'coarse_grained_metabolism'  (local:KineticMetabolism)
spec_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['vmax'] = 1.0
spec_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['km'] = 0.5
spec_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['biomass_yield'] = 0.5
spec_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['energy_yield'] = 0.3
spec_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['entropy_rate'] = 0.1
spec_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['secretion_frac'] = 0.2
spec_meta_modelers_guide_composites_fig06_executable_kinetic['state']['coarse_grained_metabolism']['config']['interval'] = 1.0

# process 'catalyzed_reaction_network'  (local:KineticReactionNetwork)
spec_meta_modelers_guide_composites_fig06_executable_kinetic['state']['catalyzed_reaction_network']['config']['k'] = 0.2
spec_meta_modelers_guide_composites_fig06_executable_kinetic['state']['catalyzed_reaction_network']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig04b-executable`** — `spec_meta_modelers_guide_composites_fig04b_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig04b_executable = load_spec(REPO / 'meta_modelers_guide/composites/fig04b-executable.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig04b_executable)

In [ ]:
# === Edit parameters for composite 'fig04b-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'cell'  (local:CellularInterfaceHandler)
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['uptake_rate'] = 0.8
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['growth_max'] = 0.6
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['km'] = 0.5
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['shape_growth_coupling'] = 1.0
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['objective_yield'] = 0.5
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['death_Ea'] = 300000.0
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['gas_R'] = 8.314
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['d_value_ref_min'] = 1.0
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['temp_ref_death'] = 55.0
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['temp_opt'] = 37.0
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['viability_init'] = 1.0
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['elasticity'] = 0.1
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['membrane_conductance'] = 0.05
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['thermal_conductance'] = 0.02
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['signaling_gain'] = 0.4
spec_meta_modelers_guide_composites_fig04b_executable['state']['cell']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig05-executable`** — `spec_meta_modelers_guide_composites_fig05_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig05_executable = load_spec(REPO / 'meta_modelers_guide/composites/fig05-executable.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig05_executable)

In [ ]:
# === Edit parameters for composite 'fig05-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'reaction_diffusion'  (local:SpatialDiffusion)
spec_meta_modelers_guide_composites_fig05_executable['state']['reaction_diffusion']['config']['diffusivity'] = 0.2
spec_meta_modelers_guide_composites_fig05_executable['state']['reaction_diffusion']['config']['interval'] = 1.0

# process 'production_degradation'  (local:ProductionDegradationField)
spec_meta_modelers_guide_composites_fig05_executable['state']['production_degradation']['config']['source_index'] = 0
spec_meta_modelers_guide_composites_fig05_executable['state']['production_degradation']['config']['source_rate'] = 0.05
spec_meta_modelers_guide_composites_fig05_executable['state']['production_degradation']['config']['decay_rate'] = 0.01
spec_meta_modelers_guide_composites_fig05_executable['state']['production_degradation']['config']['interval'] = 1.0

# process 'mechanical_stress'  (local:MechanicalRelax)
spec_meta_modelers_guide_composites_fig05_executable['state']['mechanical_stress']['config']['relax_rate'] = 0.3
spec_meta_modelers_guide_composites_fig05_executable['state']['mechanical_stress']['config']['interval'] = 1.0

# process 'single_cell_processes'  (local:SingleCellSpatial)
spec_meta_modelers_guide_composites_fig05_executable['state']['single_cell_processes']['config']['cell_index'] = 4
spec_meta_modelers_guide_composites_fig05_executable['state']['single_cell_processes']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig06-executable-fba`** — `spec_meta_modelers_guide_composites_fig06_executable_fba` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig06_executable_fba = load_spec(REPO / 'meta_modelers_guide/composites/fig06-executable-fba.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig06_executable_fba)

In [ ]:
# === Edit parameters for composite 'fig06-executable-fba' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'coarse_grained_metabolism'  (local:FBAMetabolism)
spec_meta_modelers_guide_composites_fig06_executable_fba['state']['coarse_grained_metabolism']['config']['uptake_scale'] = 10.0
spec_meta_modelers_guide_composites_fig06_executable_fba['state']['coarse_grained_metabolism']['config']['max_uptake'] = 20.0
spec_meta_modelers_guide_composites_fig06_executable_fba['state']['coarse_grained_metabolism']['config']['o2_bound'] = 18.0
spec_meta_modelers_guide_composites_fig06_executable_fba['state']['coarse_grained_metabolism']['config']['interval'] = 1.0

# process 'catalyzed_reaction_network'  (local:KineticReactionNetwork)
spec_meta_modelers_guide_composites_fig06_executable_fba['state']['catalyzed_reaction_network']['config']['k'] = 0.2
spec_meta_modelers_guide_composites_fig06_executable_fba['state']['catalyzed_reaction_network']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig07-executable`** — `spec_meta_modelers_guide_composites_fig07_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig07_executable = load_spec(REPO / 'meta_modelers_guide/composites/fig07-executable.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig07_executable)

In [ ]:
# === Edit parameters for composite 'fig07-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'molecular_mechanism'  (local:MolecularMechanismHandler)
spec_meta_modelers_guide_composites_fig07_executable['state']['molecular_mechanism']['config']['k_cat_atp'] = 100.0
spec_meta_modelers_guide_composites_fig07_executable['state']['molecular_mechanism']['config']['n_protons_per_atp'] = 3.3
spec_meta_modelers_guide_composites_fig07_executable['state']['molecular_mechanism']['config']['pmf_volts'] = 0.15
spec_meta_modelers_guide_composites_fig07_executable['state']['molecular_mechanism']['config']['torque_pn_nm'] = 40.0
spec_meta_modelers_guide_composites_fig07_executable['state']['molecular_mechanism']['config']['c_ring'] = 10.0
spec_meta_modelers_guide_composites_fig07_executable['state']['molecular_mechanism']['config']['efficiency'] = 0.75
spec_meta_modelers_guide_composites_fig07_executable['state']['molecular_mechanism']['config']['proton_charge'] = 1.602e-19
spec_meta_modelers_guide_composites_fig07_executable['state']['molecular_mechanism']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig08-executable`** — `spec_meta_modelers_guide_composites_fig08_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig08_executable = load_spec(REPO / 'meta_modelers_guide/composites/fig08-executable.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig08_executable)

In [ ]:
# === Edit parameters for composite 'fig08-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'transmembrane_transport'  (local:TransmembraneTransportODE)
spec_meta_modelers_guide_composites_fig08_executable['state']['transmembrane_transport']['config']['k'] = 0.3
spec_meta_modelers_guide_composites_fig08_executable['state']['transmembrane_transport']['config']['metabolite_frac'] = 0.1
spec_meta_modelers_guide_composites_fig08_executable['state']['transmembrane_transport']['config']['interval'] = 1.0

# process 'replication_and_repair'  (local:ReplicationAndRepairODE)
spec_meta_modelers_guide_composites_fig08_executable['state']['replication_and_repair']['config']['doubling_time_s'] = 1800.0
spec_meta_modelers_guide_composites_fig08_executable['state']['replication_and_repair']['config']['dna_init'] = 1.0
spec_meta_modelers_guide_composites_fig08_executable['state']['replication_and_repair']['config']['interval'] = 1.0

# process 'cell_metabolism'  (local:CellMetabolismODE)
spec_meta_modelers_guide_composites_fig08_executable['state']['cell_metabolism']['config']['k'] = 0.25
spec_meta_modelers_guide_composites_fig08_executable['state']['cell_metabolism']['config']['metabolite_yield'] = 0.6
spec_meta_modelers_guide_composites_fig08_executable['state']['cell_metabolism']['config']['energy_yield'] = 0.4
spec_meta_modelers_guide_composites_fig08_executable['state']['cell_metabolism']['config']['interval'] = 1.0

# process 'transcription'  (local:TranscriptionODE)
spec_meta_modelers_guide_composites_fig08_executable['state']['transcription']['config']['elong_nt_per_s'] = 45.0
spec_meta_modelers_guide_composites_fig08_executable['state']['transcription']['config']['gene_length_nt'] = 1000.0
spec_meta_modelers_guide_composites_fig08_executable['state']['transcription']['config']['mrna_halflife_s'] = 180.0
spec_meta_modelers_guide_composites_fig08_executable['state']['transcription']['config']['interval'] = 1.0

# process 'translation'  (local:TranslationODE)
spec_meta_modelers_guide_composites_fig08_executable['state']['translation']['config']['elong_aa_per_s'] = 15.0
spec_meta_modelers_guide_composites_fig08_executable['state']['translation']['config']['protein_length_aa'] = 300.0
spec_meta_modelers_guide_composites_fig08_executable['state']['translation']['config']['doubling_time_s'] = 1800.0
spec_meta_modelers_guide_composites_fig08_executable['state']['translation']['config']['km_met'] = 0.5
spec_meta_modelers_guide_composites_fig08_executable['state']['translation']['config']['km_rib'] = 0.5
spec_meta_modelers_guide_composites_fig08_executable['state']['translation']['config']['interval'] = 1.0

# process 'subunit_assembly'  (local:SubunitAssemblyODE)
spec_meta_modelers_guide_composites_fig08_executable['state']['subunit_assembly']['config']['k'] = 0.2
spec_meta_modelers_guide_composites_fig08_executable['state']['subunit_assembly']['config']['doubling_time_s'] = 1800.0
spec_meta_modelers_guide_composites_fig08_executable['state']['subunit_assembly']['config']['ribosome_init'] = 0.5
spec_meta_modelers_guide_composites_fig08_executable['state']['subunit_assembly']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig09a-executable`** — `spec_meta_modelers_guide_composites_fig09a_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig09a_executable = load_spec(REPO / 'meta_modelers_guide/composites/fig09a-executable.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig09a_executable)

In [ ]:
# === Edit parameters for composite 'fig09a-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'metabolism_closure'  (local:MetabolismClosureODE)
spec_meta_modelers_guide_composites_fig09a_executable['state']['metabolism_closure']['config']['metabolite_yield'] = 0.6
spec_meta_modelers_guide_composites_fig09a_executable['state']['metabolism_closure']['config']['entropy_rate'] = 0.1
spec_meta_modelers_guide_composites_fig09a_executable['state']['metabolism_closure']['config']['interval'] = 1.0

# process 'autocatalysis'  (local:AutocatalysisODE)
spec_meta_modelers_guide_composites_fig09a_executable['state']['autocatalysis']['config']['k'] = 0.3
spec_meta_modelers_guide_composites_fig09a_executable['state']['autocatalysis']['config']['k_cat'] = 0.15
spec_meta_modelers_guide_composites_fig09a_executable['state']['autocatalysis']['config']['interval'] = 1.0

# process 'containment_closure'  (local:ContainmentClosureODE)
spec_meta_modelers_guide_composites_fig09a_executable['state']['containment_closure']['config']['assembly_rate'] = 0.15
spec_meta_modelers_guide_composites_fig09a_executable['state']['containment_closure']['config']['perm_max'] = 0.8
spec_meta_modelers_guide_composites_fig09a_executable['state']['containment_closure']['config']['perm_km'] = 1.0
spec_meta_modelers_guide_composites_fig09a_executable['state']['containment_closure']['config']['interval'] = 1.0

# process 'membrane_self_assembly'  (local:MembraneSelfAssemblyODE)
spec_meta_modelers_guide_composites_fig09a_executable['state']['membrane_self_assembly']['config']['k'] = 0.2
spec_meta_modelers_guide_composites_fig09a_executable['state']['membrane_self_assembly']['config']['interval'] = 1.0

# process 'lipid_aggregation'  (local:LipidAggregationODE)
spec_meta_modelers_guide_composites_fig09a_executable['state']['lipid_aggregation']['config']['k'] = 0.1
spec_meta_modelers_guide_composites_fig09a_executable['state']['lipid_aggregation']['config']['interval'] = 1.0

# process 'replication_closure'  (local:ReplicationClosureODE)
spec_meta_modelers_guide_composites_fig09a_executable['state']['replication_closure']['config']['k'] = 0.2
spec_meta_modelers_guide_composites_fig09a_executable['state']['replication_closure']['config']['interval'] = 1.0

# process 'template_replication'  (local:TemplateReplicationODE)
spec_meta_modelers_guide_composites_fig09a_executable['state']['template_replication']['config']['k'] = 0.15
spec_meta_modelers_guide_composites_fig09a_executable['state']['template_replication']['config']['interval'] = 1.0

# process 'template_directed_synthesis'  (local:TemplateDirectedSynthesisProc)
spec_meta_modelers_guide_composites_fig09a_executable['state']['template_directed_synthesis']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig09b-executable`** — `spec_meta_modelers_guide_composites_fig09b_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig09b_executable = load_spec(REPO / 'meta_modelers_guide/composites/fig09b-executable.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig09b_executable)

In [ ]:
# === Edit parameters for composite 'fig09b-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'minimal_cell_containment'  (local:ContainmentODE)
spec_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_containment']['config']['assembly_rate'] = 0.15
spec_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_containment']['config']['interval'] = 1.0

# process 'minimal_cell_metabolism'  (local:MetabolismLinear)
spec_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_metabolism']['config']['k_cat'] = 0.2
spec_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_metabolism']['config']['metabolite_yield'] = 0.6
spec_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_metabolism']['config']['energy_yield'] = 0.4
spec_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_metabolism']['config']['interval'] = 1.0

# process 'gene_expression'  (local:GeneExpressionODE)
spec_meta_modelers_guide_composites_fig09b_executable['state']['gene_expression']['config']['k_expr'] = 0.25
spec_meta_modelers_guide_composites_fig09b_executable['state']['gene_expression']['config']['protein_yield'] = 0.5
spec_meta_modelers_guide_composites_fig09b_executable['state']['gene_expression']['config']['enzyme_yield'] = 0.3
spec_meta_modelers_guide_composites_fig09b_executable['state']['gene_expression']['config']['interval'] = 1.0

# process 'minimal_cell_replication'  (local:ReplicationODE)
spec_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_replication']['config']['k_rep'] = 0.1
spec_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_replication']['config']['gene_yield'] = 0.5
spec_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_replication']['config']['nucleic_yield'] = 0.4
spec_meta_modelers_guide_composites_fig09b_executable['state']['minimal_cell_replication']['config']['interval'] = 1.0

# process 'diffusion'  (local:DiffusionRelax)
spec_meta_modelers_guide_composites_fig09b_executable['state']['diffusion']['config']['turnover_rate'] = 0.05
spec_meta_modelers_guide_composites_fig09b_executable['state']['diffusion']['config']['interval'] = 1.0

# process 'reactions'  (local:MassActionReactions)
spec_meta_modelers_guide_composites_fig09b_executable['state']['reactions']['config']['k_react'] = 0.15
spec_meta_modelers_guide_composites_fig09b_executable['state']['reactions']['config']['protein_yield'] = 0.4
spec_meta_modelers_guide_composites_fig09b_executable['state']['reactions']['config']['nucleic_turnover'] = 0.2
spec_meta_modelers_guide_composites_fig09b_executable['state']['reactions']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig10-1-executable`** — `spec_meta_modelers_guide_composites_fig10_1_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig10_1_executable = load_spec(REPO / 'meta_modelers_guide/composites/fig10-1-executable.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig10_1_executable)

In [ ]:
# === Edit parameters for composite 'fig10-1-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'dna_replication'  (local:DNAReplicationODE)
spec_meta_modelers_guide_composites_fig10_1_executable['state']['dna_replication']['config']['k'] = 0.15
spec_meta_modelers_guide_composites_fig10_1_executable['state']['dna_replication']['config']['interval'] = 1.0

# process 'segregate_chromosome'  (local:SegregateChromosomeProc)
spec_meta_modelers_guide_composites_fig10_1_executable['state']['segregate_chromosome']['config']['seg_rate'] = 0.2
spec_meta_modelers_guide_composites_fig10_1_executable['state']['segregate_chromosome']['config']['interval'] = 1.0

# process 'divide'  (local:DivisionRewrite)
spec_meta_modelers_guide_composites_fig10_1_executable['state']['divide']['config']['division_time'] = 5.0
spec_meta_modelers_guide_composites_fig10_1_executable['state']['divide']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig10-2-executable`** — `spec_meta_modelers_guide_composites_fig10_2_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig10_2_executable = load_spec(REPO / 'meta_modelers_guide/composites/fig10-2-executable.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig10_2_executable)

In [ ]:
# === Edit parameters for composite 'fig10-2-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'surface_attachment'  (local:SurfaceAttachmentProc)
spec_meta_modelers_guide_composites_fig10_2_executable['state']['surface_attachment']['config']['attach_rate'] = 0.15
spec_meta_modelers_guide_composites_fig10_2_executable['state']['surface_attachment']['config']['adhesion_coef'] = 0.5
spec_meta_modelers_guide_composites_fig10_2_executable['state']['surface_attachment']['config']['interval'] = 1.0

# process 'ecm_secretion'  (local:ECMSecretionProc)
spec_meta_modelers_guide_composites_fig10_2_executable['state']['ecm_secretion']['config']['k'] = 0.2
spec_meta_modelers_guide_composites_fig10_2_executable['state']['ecm_secretion']['config']['interval'] = 1.0

# process 'biofilm_growth'  (local:BiofilmGrowthProc)
spec_meta_modelers_guide_composites_fig10_2_executable['state']['biofilm_growth']['config']['mass_rate'] = 0.25
spec_meta_modelers_guide_composites_fig10_2_executable['state']['biofilm_growth']['config']['growth_rate'] = 0.08
spec_meta_modelers_guide_composites_fig10_2_executable['state']['biofilm_growth']['config']['interval'] = 1.0

**Composite `meta_modelers_guide.composites.fig10-3-executable`** — `spec_meta_modelers_guide_composites_fig10_3_executable` (a plain, editable dict)


In [ ]:
from viva_superpowers.composite_spec import load_spec
spec_meta_modelers_guide_composites_fig10_3_executable = load_spec(REPO / 'meta_modelers_guide/composites/fig10-3-executable.composite.json')
describe_spec(spec_meta_modelers_guide_composites_fig10_3_executable)

In [ ]:
# === Edit parameters for composite 'fig10-3-executable' ===
# Each line is the spec's CURRENT value — change any, then run the Run cell
# below. The spec is a plain dict, so you may also add or remove keys.

# process 'variation'  (local:VariationProc)
spec_meta_modelers_guide_composites_fig10_3_executable['state']['variation']['config']['interval'] = 1.0

# process 'selection'  (local:SelectionProc)
spec_meta_modelers_guide_composites_fig10_3_executable['state']['selection']['config']['k'] = 0.3
spec_meta_modelers_guide_composites_fig10_3_executable['state']['selection']['config']['interval'] = 1.0

# process 'port_addition'  (local:PortAdditionProc)
spec_meta_modelers_guide_composites_fig10_3_executable['state']['port_addition']['config']['onset_rate'] = 0.1
spec_meta_modelers_guide_composites_fig10_3_executable['state']['port_addition']['config']['capacity'] = 1.0
spec_meta_modelers_guide_composites_fig10_3_executable['state']['port_addition']['config']['interval'] = 1.0

### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: the-living-atlas ===
STUDY = 'the-living-atlas'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**the-living-atlas-dynamics**


In [ ]:
# the-living-atlas-dynamics
show_viz(_render_one('image:visualizations/the-living-atlas-dynamics.svg', {'chart': 'image', 'caption': "The composed whole cell — grow, divide, die — one run of wholecell.py's modules."}, RUNS_DB, STUDY_YAML))

**the-living-atlas-gallery**


In [ ]:
# the-living-atlas-gallery
show_viz(_render_one('image:visualizations/the-living-atlas-gallery.svg', {'chart': 'image', 'caption': 'All twelve compiled figures, each running on its own.'}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| cell-grows | kind=observable path=biomass expr=max(biomass) | op >= value 3.0 provenance viability-gated metabolism converts nutrient uptake to biomass; the composed cell grows biomass 0.3 → peak ≈5.1 (exec max≈5.1) |
| cell-divides | kind=observable path=cell_count expr=last(cell_count) | op >= value 2.0 provenance once biomass crosses the division threshold the rewrite fires ONCE, partitioning the cell into two daughters — cell_count 1 → 2 at t≈3.4 (exec last=2.0) |
| cell-dies | kind=observable path=viability expr=last(viability) | op <= value 0.1 provenance the thermal shock (temperature 37 → 50 °C at t≈12) leaves the viability band; the monitor collapses viability 1.0 → 0.018 and the disintegration rewrite turns biomass to debris 0 → 4.87 (exec last=0.018) |
| all-figures-run | kind=observable path=ports.chemical_out expr=last(ports.chemical_out) | op >= value 50.0 provenance all twelve fig*-executable composites compile and run to completion (12/12); fig07''s F1Fo ATP synthase charging its port chemical_out 0 → 100 is a witness that the gallery executes (exec last=100.0) |
